# Block 3 — LAB: Engineered Features & Denoising With ML Models
### Advanced Machine Learning — M&T Bank

Starts from `bank_marketing_clean.csv` (Block 2's output). By the end, you'll save `bank_marketing_features.csv`,
which carries forward into Linear Models (Block 4) and every block after it.

**Part 1 — Engineer:** bin `age`, ordinal-encode `education`, one-hot encode the remaining nominal columns,
cyclically encode `month`, derive `campaign_intensity`.

**Part 2 — Denoise:** check the macro-economic correlations, fit PCA, reconstruct the five macro columns from
the top components, compare before/after.

Look for `# TODO` — that's where your code goes. Each task has a hint; ask if you get stuck.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", 30)

NAVY = "#251E4E"
PINK = "#FF1675"
GRAY = "#6b7280"

# Data source: https://raw.githubusercontent.com/mithun-rk/mt-advml-course/main/Data/bank_marketing_clean.csv
df = pd.read_csv("https://raw.githubusercontent.com/mithun-rk/mt-advml-course/main/Data/bank_marketing_clean.csv", sep=";")
print(df.shape)
df.head()


# Part 1 — Engineered Features

### Task 1.1 — Bin `age` into life-stage bands

`age` ranges 17–98 (median 38). A raw linear term forces the model to assume risk moves in a straight line with
age, which rarely holds. Binning lets a model treat each life stage independently.

**Hint:** `pd.cut(df["age"], bins=..., labels=..., include_lowest=True)`. Try bin edges
`[17, 25, 35, 45, 55, 65, 98]` with 6 labels.


In [ ]:
# TODO: create age_bins (list of 7 edges) and age_labels (list of 6 label strings)
age_bins = None
age_labels = None

# TODO: create df["age_band"] using pd.cut(...)


df[["age", "age_band"]].sample(8, random_state=1)


### Task 1.2 — Ordinal-encode `education`

Education has a genuine order — `illiterate` is not "a different category" from `university.degree`, it's *less*
of the same thing. Ordinal encoding preserves that; one-hot encoding would throw it away.

**Hint:** build a Python dict mapping each education level to a rank (0 = lowest), then `df["education"].map(...)`.
The seven levels, from lowest to highest: `illiterate, basic.4y, basic.6y, basic.9y, high.school,
professional.course, university.degree`.


In [ ]:
# TODO: define education_order as a list from lowest to highest
education_order = None

# TODO: build education_rank_map = {level: rank, ...} from education_order
education_rank_map = None

# TODO: create df["education_rank"] using .map(education_rank_map)


df[["education", "education_rank"]].drop_duplicates().sort_values("education_rank")


### Task 1.3 — One-hot encode the remaining nominal columns

`job`, `marital`, `default`, `housing`, `loan`, `contact`, `day_of_week`, `poutcome`, and our new `age_band` have
no real order — one-hot encoding avoids implying a false ranking between, say, `"admin."` and `"technician"`.

**Hint:** `pd.get_dummies(df, columns=[...], drop_first=True)`.


In [ ]:
nominal_cols = [
    "job", "marital", "default", "housing", "loan",
    "contact", "day_of_week", "poutcome", "age_band",
]

# TODO: create df_encoded by one-hot encoding nominal_cols
df_encoded = None

print("columns before:", df.shape[1], "-> after one-hot:", df_encoded.shape[1])
df_encoded.filter(like="job_").head()


### Task 1.4 — Cyclically encode `month`

`month` wraps around — December (12) and January (1) are one month apart, not eleven. A sine/cosine pair encodes
that adjacency; a raw integer or a one-hot column can't.

**Hint:** map month abbreviation → number (1–12), then
`month_sin = sin(2 * pi * month_num / 12)`, `month_cos = cos(2 * pi * month_num / 12)`. Drop the original `month`
column once you're done with it.


In [ ]:
month_number = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
    "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}

# TODO: map df_encoded["month"] to numbers using month_number
month_num = None

# TODO: create df_encoded["month_sin"] and df_encoded["month_cos"]


# TODO: drop the original "month" column from df_encoded


df_encoded[["month_sin", "month_cos"]].drop_duplicates().round(2)


### Task 1.5 — Derive `campaign_intensity`

One number that captures "contacted a lot this campaign, with little prior success": a high value flags clients
being pushed hard despite a weak track record.

**Hint:** `campaign / (previous + 1)` — the `+ 1` avoids dividing by zero for clients with no prior contacts.


In [ ]:
# TODO: create df_encoded["campaign_intensity"]


df_encoded[["campaign", "previous", "campaign_intensity"]].describe()


**Note on `duration`:** we're leaving it as-is for now. It's the single strongest predictor in this dataset — and
also the dataset's own documented leakage trap, since call length is only known *after* the call happens. We'll
come back to it directly when we talk about realistic evaluation in Block 4.


### Task 1.6 — Encode the target

One last housekeeping step: turn `y` into a 0/1 column so every model we fit from Block 4 onward can use it directly.

**Hint:** `(df_encoded["y"] == "yes").astype(int)`


In [ ]:
# TODO: overwrite df_encoded["y"] as a 0/1 integer column


df_encoded["y"].value_counts()


# Part 2 — Denoising With ML Models (PCA)

### Task 2.1 — Check the macro-economic correlations

Five columns describe the same broad economic backdrop for every call: `emp.var.rate`, `cons.price.idx`,
`cons.conf.idx`, `euribor3m`, `nr.employed`. Let's see how independent they really are.

**Hint:** `df_encoded[macro_cols].corr()`


In [ ]:
macro_cols = ["emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]

# TODO: compute the correlation matrix of macro_cols
corr = None

corr.round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(corr, cmap="RdPu", vmin=0, vmax=1)
ax.set_xticks(range(len(macro_cols)))
ax.set_xticklabels(macro_cols, rotation=45, ha="right")
ax.set_yticks(range(len(macro_cols)))
ax.set_yticklabels(macro_cols)
for i in range(len(macro_cols)):
    for j in range(len(macro_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if corr.iloc[i, j] > 0.5 else NAVY, fontsize=9)
ax.set_title("Macro-Economic Feature Correlation", color=NAVY, fontweight="bold")
plt.tight_layout()
plt.show()


**Question to answer before moving on:** which columns look almost redundant with each other? Which one looks
more independent?


### Task 2.2 — Fit PCA and check explained variance

Standardize first (PCA is scale-sensitive), then see how many components it actually takes to capture this
group's real signal.

**Hint:** `StandardScaler().fit_transform(...)`, then `PCA().fit(...)` and look at `.explained_variance_ratio_`.


In [ ]:
# TODO: standardize df_encoded[macro_cols] into macro_scaled
scaler = StandardScaler()
macro_scaled = None

# TODO: fit a PCA() (all components) on macro_scaled
pca = None

explained = pca.explained_variance_ratio_
cumulative = explained.cumsum()

for i, (e, c) in enumerate(zip(explained, cumulative), start=1):
    print(f"PC{i}: {e:.1%} of variance  (cumulative {c:.1%})")


**Decision point:** based on the cumulative variance printed above, how many components would you keep to capture
~99% of the real signal while dropping the noise-only components? (The solution uses 3 — see if your numbers agree.)


### Task 2.3 — Reconstruct (denoise) the five columns

Project into the reduced component space, then project straight back. What comes back is the same five columns,
smoothed — the noise-only components never make it into the reconstruction.

**Hint:** `PCA(n_components=k)`, `.fit_transform(...)` to project, `.inverse_transform(...)` to reconstruct, then
`scaler.inverse_transform(...)` to get back to the original scale.


In [ ]:
# TODO: set k to the number of components you decided on above
k = None

# TODO: fit a PCA(n_components=k) and project macro_scaled -> macro_projected
pca_denoise = None
macro_projected = None

# TODO: inverse_transform macro_projected back to macro_reconstructed_scaled,
#       then scaler.inverse_transform(...) to get macro_reconstructed
macro_reconstructed_scaled = None
macro_reconstructed = None

for i, col in enumerate(macro_cols):
    df_encoded[col + "_denoised"] = macro_reconstructed[:, i]

df_encoded[["euribor3m", "euribor3m_denoised", "cons.conf.idx", "cons.conf.idx_denoised"]].describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sample = df_encoded.sample(400, random_state=1).sort_values("euribor3m")

for ax, col in zip(axes, ["euribor3m", "cons.conf.idx"]):
    ax.scatter(range(len(sample)), sample[col], s=8, color=GRAY, alpha=0.5, label="Original")
    ax.scatter(range(len(sample)), sample[col + "_denoised"], s=8, color=PINK, label="Denoised")
    ax.set_title(col, color=NAVY, fontweight="bold")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()


**Talking point:** does the denoised version track the same underlying shape as the original, with the
point-to-point jitter smoothed out? We haven't dropped any columns — a model using the `_denoised` versions still
sees five macro features, just five cleaner ones.


## Save the Engineered Feature Set

In [ ]:
# TODO: save df_encoded to "bank_marketing_features.csv", sep=";", index=False


print("Saved bank_marketing_features.csv —", df_encoded.shape)


## Recap

- **Engineering** turns a raw-but-clean column into something that matches how the underlying relationship
  actually behaves — order-preserving for ordinal data, no-order for nominal data, wraparound-aware for cyclical
  data, ratio-aware where a single derived number beats two raw ones.
- **Denoising** is for the opposite problem: a whole *group* of features that are too correlated, each carrying
  a noisy version of one shared signal. PCA reconstruction keeps every column but strips the noise-only variance
  out of them.
- `bank_marketing_features.csv` is now the shared feature set for the rest of the course.

**Up next — Block 4:** Optimizing & Training Linear Models, starting from `bank_marketing_features.csv`.
